# Esplorazione dei dati
Ricevo 'vendite.csv' e 'ordini.csv' come li riceverebbe un analista: senza sapere come essi siano stati prodotti.\
L'obiettivo è capire cosa c'è nei file e quali problemi hanno, prima di qualsiasi analisi.\
Il risultato finale è la tabella dei problemi in fondo, che diventa la lista delle regole di pulizia

## 1. Caricamento
Importa pandas e legge i due file CSV dalla cartella /data. Il .. vuol dire "sali di una cartella": il notebook sta in notebooks/, i dati in data/.

In [1]:
import pandas as pd
vendite = pd.read_csv("../data/vendite.csv")
ordini = pd.read_csv("../data/ordini.csv")

## 2. Dimensioni 
.shape restituisce righe e colonne\
è il primo numero da annotare, ogni pulizia dovrà dire quante righe toglie rispetto a questo totale.\
Vendite = 16569 righe x 20 colonne\
Ordini = 4514 righe x 16 colonne

In [2]:
print(vendite.shape)
print(ordini.shape)

(16569, 20)
(4514, 16)


## 3. Struttura delle vendite
.info() mostra per ogni colonna il tipo di dato e quanti valori non sono vuoti.\
Controllare il tipo di dato per ogni colonna ed il conteggio di "non-null", dove se è più basso del totale, la colonna ha valori mancanti.\
data_ora e prezzo_venduto sono lette come testo, ma dovrebbero essere una data ed un numero, deduco che c'è qualcosa che impedisce la conversione. Inoltre diverse colonne hanno meno valori del totale.

In [3]:
vendite.info()

<class 'pandas.DataFrame'>
RangeIndex: 16569 entries, 0 to 16568
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_riga           16569 non-null  int64  
 1   data_ora          16569 non-null  str    
 2   scontrino         16569 non-null  str    
 3   articolo          16476 non-null  str    
 4   brand             16476 non-null  str    
 5   categoria         16476 non-null  str    
 6   taglia            16476 non-null  str    
 7   colore            16476 non-null  str    
 8   stagione          16476 non-null  str    
 9   prezzo_listino    16569 non-null  float64
 10  prezzo_venduto    16487 non-null  str    
 11  sconto_pct        16569 non-null  int64  
 12  motivo_sconto     16486 non-null  str    
 13  tipo_movimento    16569 non-null  str    
 14  rif_vendita       484 non-null    float64
 15  esito_reso        484 non-null    str    
 16  motivo_reso       484 non-null    str    
 17  meto

## 4. Valori mancanti
.isna() dice vero/falso per ogni cella vuota, .sum() conta i vuoti per ogni colonna. Il filtro mancanti > 0 mostra solo le colonne con almeno un vuoto.\
Un vuoto non è per forza un errore, piuttosto è importante chiedersi se in quella riga il campo dovrebbe esserci.

In [4]:
mancanti = vendite.isna().sum()
mancanti[mancanti>0]

articolo               93
brand                  93
categoria              93
taglia                 93
colore                 93
stagione               93
prezzo_venduto         82
motivo_sconto          83
rif_vendita         16085
esito_reso          16085
motivo_reso         16085
metodo_pagamento      484
id_buono            15844
reso_fornitore      16481
dtype: int64

## 5. Valori mancanti per tipo di movimento
Divide le righe per tipo di movimento tra vendita, reso e buono regalo, poi conta i vuoti in ciascun gruppo. Lamdba d: ... è una funzione 'usa e getta' senza nome, applicata ad ogni gruppo.\
Qui si distinguono i vuoti giusti da quelli sbagliati.\
Sono corretti i vuoti per articolo, brand e taglia sui 93 buoni regalo, poi i metodi di pagamento vuoti per i 484 resi ed infine i campi del reso vuoti sulle vendite.\
Mentre sono errati i 81 prezzo_venduto vuoti su vendite e resi, e gli 83 motivo_sconto vuoti.

In [5]:
vendite.groupby("tipo_movimento").apply(lambda d: d.isna().sum())

,id_riga,data_ora,scontrino,articolo,brand,categoria,taglia,colore,stagione,prezzo_listino,prezzo_venduto,sconto_pct,motivo_sconto,rif_vendita,esito_reso,motivo_reso,metodo_pagamento,id_buono,reso_fornitore
tipo_movimento,,,,,,,,,,,,,,,,,,,
buono_regalo,0,0,0,93,93,93,93,93,93,0,1,0,0,93,93,93,0,0,93
reso,0,0,0,0,0,0,0,0,0,0,3,0,2,0,0,0,484,162,396
vendita,0,0,0,0,0,0,0,0,0,0,78,0,81,15992,15992,15992,0,15682,15992


## 6. Righe duplicate
.duplicated() segna le righe identiche a una riga precedente. keep=False le segna tutte e due, così le vedo una accanto all'altra.\
Se anche l'id_riga si ripete, non sono due vendite uguali ma la stessa vendita registrata due volte, per esempio da un doppio export.\
164 righe duplicate, con lo stesso id_riga. Di cui tenere una sola, altrimenti l'incasso risulterebbe gonfiato.

In [6]:
print("Righe identitiche:", vendite.duplicated().sum())
print("id_riga ripetuti:", vendite["id_riga"].duplicated().sum())
vendite[vendite.duplicated(keep=False)].head(6)

Righe identitiche: 164
id_riga ripetuti: 164


,id_riga,data_ora,scontrino,articolo,brand,categoria,taglia,colore,stagione,prezzo_listino,prezzo_venduto,sconto_pct,motivo_sconto,tipo_movimento,rif_vendita,esito_reso,motivo_reso,metodo_pagamento,id_buono,reso_fornitore
12,13,2023-03-05 17:25:00,S2023-000009,GRI-CAL-0007,Grillo,calzini,43-46,blu,PE23,39.0,39.0,0,nessuno,vendita,NaN,NaN,NaN,carta,NaN,NaN
13,13,2023-03-05 17:25:00,S2023-000009,GRI-CAL-0007,Grillo,calzini,43-46,blu,PE23,39.0,39.0,0,nessuno,vendita,NaN,NaN,NaN,carta,NaN,NaN
74,74,2023-03-16 17:04:00,S2023-000052,FRE-GIA-0009,FreePlace Outerwear,giacca,48,grigio,PE23,219.0,219.0,0,nessuno,vendita,NaN,NaN,NaN,carta,NaN,NaN
75,74,2023-03-16 17:04:00,S2023-000052,FRE-GIA-0009,FreePlace Outerwear,giacca,48,grigio,PE23,219.0,219.0,0,nessuno,vendita,NaN,NaN,NaN,carta,NaN,NaN
414,413,2023-05-09 17:28:00,S2023-000305,GRI-CAL-0002,Grillo,calzini,39-42,rosso,PE23,39.0,35.1,10,cliente_abituale,vendita,NaN,NaN,NaN,carta,NaN,NaN
415,413,2023-05-09 17:28:00,S2023-000305,GRI-CAL-0002,Grillo,calzini,39-42,rosso,PE23,39.0,35.1,10,cliente_abituale,vendita,NaN,NaN,NaN,carta,NaN,NaN


## 7. Valori distinti delle colonne di testo
Per ogni colonna .value_counts() conta quante volte compare ogni valore. dropna=False conta anche i vuoti. set_option alza il numero di righe mostrate, altrimenti l'output viene tagliato con ... proprio dove stanno i valori strani.\
In un elenco chiuso, per esempio le taglie, dovrebbero comparire in un solo modo. Le righe con conteggi piccoli in fondo sono quasi sempre errori di scrittura.\
Le taglie vengono spesso scritte in diverso modo.\
I brand con maiuscole, minuscole e spazi di troppo.\
Per il colore, vale lo stesso problema dei brand, con l'aggiunta di un colore il blu navy.

In [7]:
pd.set_option("display.max_rows", 200)
for col in ["taglia", "brand", "colore", "motivo_sconto", "tipo_movimento", "metodo_pagamento"]:
    print(f"---{col}---")
    print(vendite[col].value_counts(dropna=False))

---taglia---
taglia
TU             2198
M              2052
L              2031
50             1321
48             1070
52             1025
S               888
XL              874
43-46           841
39-42           649
54              484
46              457
44              418
43              393
42              383
XXL             381
45              176
41              168
56              166
XS              111
NaN              93
M                29
m                26
L                25
Large            22
l                21
Medium           18
50               16
tg 52            16
tg 48            16
tg 50            16
S                15
XL               14
Extra Large      14
52               14
xl               12
Small             9
tg 43-46          9
tg 46             8
tg 39-42          8
39-42             8
48                8
43-46             6
s                 6
44                6
tg 43             5
tg 54             4
tg XXL            4
xxl               4


## 8. Statistiche dei numeri
.describe() calcola per ogni colonna numerica: conteggio, media, deviazione standard, minimo, quartili e massimo.\
Si guardano sopratutto min e max, dicono subito se ci sono valori impossibili.\
sconto_pct va da -10 a 300, sono errori da correggere.

In [8]:
vendite.describe()

,id_riga,prezzo_listino,sconto_pct,rif_vendita
count,16569.000000,16569.000000,16569.000000,484.000000
mean,8202.249562,143.728891,26.192890,8192.508264
std,4734.341045,108.281964,25.992605,4840.862325
min,1.000000,39.000000,-10.000000,55.000000
25%,4106.000000,69.000000,0.000000,3973.250000
50%,8197.000000,109.000000,30.000000,8044.000000
75%,12302.000000,159.000000,50.000000,12524.000000
max,16405.000000,589.000000,300.000000,16329.000000


## 9. Perché il prezzo è testo
pd.to_numeric (....., errors="coerce") prova a trasformare ogni valore in un numero, se non ci riesce mette un vuoto. Poi si mostrano i valori che hanno fallito.\
Quello che compare nell'elenco è ciò che blocca la conversione.\
Alcuni prezzo vengono scritti con la virgola, mentre python vuole il punto. Basterà sostituirlo

In [9]:
pd.to_numeric(vendite["prezzo_venduto"], errors="coerce").isna().sum()
vendite.loc[pd.to_numeric(vendite["prezzo_venduto"], errors="coerce").isna(), "prezzo_venduto"].dropna().head(10)

24      59,00
57     219,00
119     59,00
123    369,00
181    139,00
254     39,00
329    129,00
351    239,00
475     59,00
483    197,10
Name: prezzo_venduto, dtype: str

## 10. Perché la data è testo
.str.len() misura la lunghezza di ogni data scritta come testo, value.counts() conta invece quante date hanno ogni lunghezza.\
Se tutte le date fossero scritte allo stesso modo, avrebbero tutte la stessa lunghezza.\
16071 date da 19 caratteri ed altre 498 da 16; due formati diversi nella stessa colonna.

In [10]:
vendite["data_ora"].str.len().value_counts()

data_ora
19    16071
16      498
Name: count, dtype: int64

## 11. Altri controlli sulle vendite
Tre controlli che da solo .describe() non fa.\
Converte il prezzo (con la virgola sostituita) e conta quelli negativi.\
Converte le date accettando entrambi i formati e conta quelle fuori dal periodo dei dati.\
Ricalcola lo sconto da listino e prezzo venduta, e conta le righe dove non coincide con sconto_pct.

In [11]:
prezzo = pd.to_numeric(vendite["prezzo_venduto"].str.replace(",", "."), errors="coerce")
print("Prezzi negativi:", (prezzo < 0).sum())

data = pd.to_datetime(vendite["data_ora"], format="mixed", dayfirst=True)
print("Date fuori dal periodo 2023-2026:", ((data.dt.year < 2023) | (data.dt.year > 2026)).sum())

atteso_sconto = ((1 - prezzo / vendite["prezzo_listino"]) * 100).round()
print("Sconto che non torna con i prezzi:", ((atteso_sconto - vendite["sconto_pct"]).abs() > 1).sum())

Prezzi negativi: 8
Date fuori dal periodo 2023-2026: 3
Sconto che non torna con i prezzi: 95


## 12. Struttura degli ordini
.info() mostra per ogni colonna il tipo di dato e quanti valori non sono vuoti.\
Nessun valore mancante.\
data_ordine e data_arrivo sono lette come testo, andranno convertite in date.

In [12]:
ordini.info()

<class 'pandas.DataFrame'>
RangeIndex: 4514 entries, 0 to 4513
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_ordine          4514 non-null   int64  
 1   articolo           4514 non-null   str    
 2   taglia             4514 non-null   str    
 3   brand              4514 non-null   str    
 4   fornitore          4514 non-null   str    
 5   modalita_acquisto  4514 non-null   str    
 6   categoria          4514 non-null   str    
 7   colore             4514 non-null   str    
 8   stagione           4514 non-null   str    
 9   tipo_ordine        4514 non-null   str    
 10  data_ordine        4514 non-null   str    
 11  data_arrivo        4514 non-null   str    
 12  qta_ordinata       4514 non-null   int64  
 13  qta_ricevuta       4514 non-null   int64  
 14  costo_unitario     4514 non-null   float64
 15  prezzo_listino     4514 non-null   float64
dtypes: float64(2), int64(3), str(11)
me

## 13. Duplicati degli ordini
3 righe duplicate, con lo stesso id_ordine. Da tenerne una.

In [13]:
print("Righe identiche:", ordini.duplicated().sum())
print("id_ordine ripetuti:", ordini["id_ordine"].duplicated().sum())

Righe identiche: 3
id_ordine ripetuti: 3


## 14. Statistiche degli ordini
Min e max per quantità e prezzi.\
Nessun valore negativo. qta_ricevuta può essere 0, non rappresenta un errore di per sé.

In [14]:
ordini.describe()

,id_ordine,qta_ordinata,qta_ricevuta,costo_unitario,prezzo_listino
count,4514.000000,4514.000000,4514.000000,4514.000000,4514.000000
mean,2255.674790,4.703367,4.661498,75.424878,175.220647
std,1302.487182,4.333530,4.346013,53.663129,123.237339
min,1.000000,1.000000,0.000000,13.270000,39.000000
25%,1128.250000,2.000000,2.000000,38.120000,89.000000
50%,2255.500000,3.000000,3.000000,55.960000,129.000000
75%,3383.750000,6.000000,6.000000,106.600000,239.000000
max,4511.000000,31.000000,31.000000,370.800000,589.000000


## 15. Controlli di coerenza negli ordini
3 domande: si può ricevere più pezzi di quelli che ordini?, un capo può costare al negozio più di quanto lo vende?, la merce può arrivare prima di averla ordinata?\
In tutti e 3 i casi la risposta giusta è no, quindi ogni riga contata è un errore.\
5 righe con più pezzi ricevuti di quelli ordinati.\
3 con il costo sopra al prezzo di listino.\
3 con l'arrivo prima dell'ordine.

In [15]:
print("Ricevuti più degli ordinati:", (ordini["qta_ricevuta"] > ordini["qta_ordinata"]).sum())
print("Costo più alto del listino:", (ordini["costo_unitario"] > ordini["prezzo_listino"]).sum())
date_ordine = pd.to_datetime(ordini["data_ordine"])
date_arrivo = pd.to_datetime(ordini["data_arrivo"])
print("Arrivo prima dell'ordine:", (date_arrivo < date_ordine).sum())

Ricevuti più degli ordinati: 5
Costo più alto del listino: 3
Arrivo prima dell'ordine: 3


## 16. I due file sono coerenti tra loro?
Confronta gli insiemi di codici articolo, un articolo venduto ma assente dagli ordini è un codice orfano: il negozio non può vendere ciò che non ha mai comprato.\
Per ogni riga degli ordini, va a prendere il listino dello stesso articolo nelle vendite e controlla che coincida.\
Un file può essere pulito da solo ma in contraddizione con l'altro.\
8 articoli venduti ma mai ordinati.

In [16]:
venduti = set(vendite["articolo"].dropna())
ordinati = set(ordini["articolo"])
print("Articoli venduti mai ordinati:", len(venduti - ordinati))
print(sorted(venduti - ordinati))

listino_v = vendite.groupby("articolo")["prezzo_listino"].first()
atteso = ordini["articolo"].map(listino_v)
print("Righe di ordini con listino diverso:", (atteso.notna() & (ordini["prezzo_listino"] != atteso)).sum())

Articoli venduti mai ordinati: 8
['AGO-CAM-9128', 'AGO-CAM-9604', 'AGO-CRA-9297', 'AGO-CRA-9988', 'CRU-MAG-9269', 'DIP-FEL-9335', 'LID-PAN-9059', 'LID-PAN-9513']
Righe di ordini con listino diverso: 5


## Problemi trovati — vendite.csv
| Colonna | Problema | Quante righe | Cosa fare |
|---|---|---|---|
| tutte | righe duplicate, stesso id_riga | 164 | pulire: tenerne una |
| data_ora | due formati: 2024-01-05 17:32:00 e 05/01/2024 17:32 | 498 | pulire: convertire entrambi in data |
| data_ora | anni fuori dal periodo 2023–2026 | [ ] | pulire: escludere |
| prezzo_venduto | numeri con la virgola, letti come testo | ~330 | pulire: virgola → punto |
| prezzo_venduto | mancante su vendite e resi | 81 | pulire: escludere dai calcoli sui prezzi |
| prezzo_venduto | negativi | [ ] | pulire: escludere |
| sconto_pct | fuori range (min −10, max 300) | [ ] | pulire: ricalcolare dai prezzi |
| sconto_pct | non torna con listino e prezzo venduto | [ ] | pulire: ricalcolare dai prezzi |
| motivo_sconto | mancante con sconto presente | 83 | pulire: dedurlo (saldo se in periodo di saldi) o segnare "sconosciuto" |
| taglia | stessa taglia scritta in modi diversi (m, Medium, tg 48, spazi) | ~490 | pulire: uniformare |
| brand | maiuscole, minuscole, spazi finali | ~330 | pulire: uniformare |
| colore | maiuscole e "blu navy" fuori dall'elenco | ~160 | pulire: uniformare, blu navy → blu |
| articolo | 8 articoli venduti mai ordinati (codici 9xxx) | 8 | pulire: escludere dalle analisi con gli ordini, segnalare |



## Problemi trovati — ordini.csv
| Colonna | Problema | Quante righe | Cosa fare |
|---|---|---|---|
| tutte | righe duplicate, stesso id_ordine | 3 | pulire: tenerne una |
| data_ordine, data_arrivo | lette come testo | tutte | pulire: convertire in data |
| data_arrivo | arrivo prima dell'ordine | 3 | pulire: segnalare ed escludere dai calcoli sui tempi |
| qta_ricevuta | più pezzi ricevuti che ordinati | 5 | pulire: segnalare, usare qta_ordinata come massimo |
| costo_unitario | costo più alto del listino | 3 | pulire: segnalare, escludere dal margine |
| prezzo_listino | diverso dal listino delle vendite | [ ] | pulire: vale il listino delle vendite |